# JSON 05 - When it breaks, and a real messy file

Two halves: first make `json` fail on purpose so the error messages stop scaring
you, then handle a file with holes in it.

## Part A - `JSONDecodeError`

Run each of these and **read the message**:

```python
json.loads("{'name': 'Amine'}")   # single quotes
json.loads('{"a": 1,}')           # trailing comma
json.loads('{name: "Amine"}')     # key not quoted
json.loads('{"x": None}')         # Python None instead of null
json.loads('')                    # empty
```

**Expected messages:**

```
single quotes   -> Expecting property name enclosed in double quotes (line 1 col 2)
trailing comma  -> Expecting property name enclosed in double quotes (line 1 col 9)
unquoted key    -> Expecting property name enclosed in double quotes (line 1 col 2)
None not null   -> Expecting value (line 1 col 7)
empty string    -> Expecting value (line 1 col 1)
```

Notice the **line and column** - that is the parser telling you exactly where it
gave up. When a real file fails to load, that number is where you look.

Now catch it properly:

```python
try:
    obj = json.loads(text)
except json.JSONDecodeError as e:
    print("bad JSON:", e.msg, "at line", e.lineno, "col", e.colno)
```

`json.JSONDecodeError` is a subclass of `ValueError`, so `except ValueError` also
catches it.

Write `safe_load(text)` that returns the object, or `None` if the text is broken,
and test it on all 5 bad strings plus one good one.

In [ ]:
import json

# TODO

## Part B - `../data/library.json`

```json
{
  "library": "Bibliothèque Centrale",
  "sections": [
    { "name": "Informatique",
      "shelves": [ { "code": "I-1", "books": [ {...}, {...} ] },
                   { "code": "I-2", "books": [ {...} ] } ] },
    { "name": "Mathématiques", "shelves": [ { "code": "M-1", "books": [ {...}, {...}, {...} ] } ] },
    { "name": "Archives" }                        <-- NO "shelves" key at all
  ]
}
```

Three levels: **section -> shelf -> book**. And two kinds of hole:
- the `Archives` section has **no `shelves` key**
- two books have **no `year`**

This is what real data looks like.

## Exercise 1 - Collect every book

Write `all_books(data)` returning a flat `list` of the book dicts, using three
nested loops (section, shelf, book). Guard the missing key with
`section.get("shelves", [])`.

**Expected: 6 books.**

Then:
- total pages -> **4272**
- titles with no `year` -> `['Python Crash Course', 'Probabilités']`
- oldest book (among those that have a year) -> `Clean Code` (**2008**)

In [ ]:
# TODO

## Exercise 2 - Books per section

**Expected:**

```python
{'Informatique': 3, 'Mathématiques': 3, 'Archives': 0}
```

`Archives` must appear with `0`, not be skipped and not crash. If your code drops
it, your `.get` default is in the wrong place.

In [ ]:
# TODO

## Exercise 3 - Find a book, and its address

`find_book(data, title)` returns `(section_name, shelf_code, book)` or `None`.

- `find_book(data, "Algorithms")` -> `('Informatique', 'I-2', {...})`
- `find_book(data, "Dune")` -> `None`

Then `where(data, "Calculus")` printing `Mathématiques / M-1 / Calculus`.

Same walk as Exercise 1 - you are just remembering *where* you were when you
found it. That "carry the path down as you descend" idea is the one you will
need for the chain-of-command exercise in the Org Chart TP.

In [ ]:
# TODO

## Exercise 4 - Clean it up and save

Produce a flat list, one dict per book, with the holes filled and the location
attached:

```python
{"title": ..., "author": ..., "pages": ..., "year": <year or None>,
 "section": ..., "shelf": ...}
```

Write it to `../data/library_flat.json` with `indent=2, ensure_ascii=False`,
then load it back and check you get 6 records.

**Flattening a nested file into clean records is 80% of real data work.** If you
can do this one comfortably, the JSON part is done.

In [ ]:
# TODO

## Recap - 05

- `json.JSONDecodeError` (a `ValueError`) tells you the exact line and column.
- Real data has holes: `.get(key, default)` at **every** level that can be missing.
- Flatten nested -> list of clean records; that is the shape everything else likes.

---

### You are done with the track

Checklist - can you do these without looking?

- [ ] `load` vs `loads` vs `dump` vs `dumps`
- [ ] `json.dump(obj, f, indent=2, ensure_ascii=False)` from memory
- [ ] walk `data["a"][0]["b"]` and say the type at each step
- [ ] `.get(key, default)` and the chained `.get("x", {}).get("y")`
- [ ] group records into `{key: [items]}`
- [ ] index records into `{key: record}` and say why
- [ ] name 3 Python things JSON cannot store
- [ ] read a `JSONDecodeError` and find the character it names

Then go back to **`tp/TP_OrgChart.ipynb`** - `company.json` is now just a nested
dict you know how to walk.